In [8]:
%run ./dev-v1.ipynb

## 4.1 localBeamSearch

Giữ beamWidth phương án song song. Mỗi vòng chỉnh nhẹ tất cả phương án
đang có, gom chung một rổ, rồi giữ lại beamWidth phương án điểm cao nhất.

beamWidth = 1 chính là Hill Climbing.


In [ ]:
def localBeamSearch(beamWidth,maxRounds=100,maxNoGain=10, seed=0,numStations=None, weights=None):
    rng = np.random.default_rng(seed)
    # tạo phương án ngẫu nhiên làm điểm xuất phát
    beam = [randomState(numStations,rng) for _ in range(beamWidth)]
    beamScores = np.array([objective(state,weights) for state in beam])
    
    # ghi dữ liệu ban đầu
    bestIndex = int(np.argmax(beamScores))
    bestState = beam[bestIndex]
    bestScore = float(beamScores[bestIndex])
    
    history = [{
        "round": 0,
        "bestScore": bestScore,
        "beamBest": float(beamScores.max()),
        "beamMean": float(beamScores.mean()),
    }]
    noGainRounds = 0
    
    # với phương án đang có, chỉnh nhẹ 1 trạm dời sang trạm khác 
    for roundNo in range(1,maxRounds+1):
        neighborPool = list({
            newState
            for state in beam
            for newState in getNeighbors(state)
        })
        
        if not neighborPool:
            break
        
        neighborScores = np.array([objective(state,weights) for state in neighborPool])
        topIndexs = np.argsort(neighborScores)[::-1][:beamWidth]
        beam = [neighborPool[i] for i in topIndexs]
        beamScores = neighborScores[topIndexs]
        
        # beamScores[0] tốt nhất của riêng vòng này
        # bestScore tốt nhất từ đầu tới giờ
        if beamScores[0] > bestScore:
            bestState = beam[0]
            bestScore = float(beamScores[0])
            noGainRounds = 0
        else:
            noGainRounds +=1 
        history.append({
            "round": roundNo,
            "bestScore": bestScore,
            "beamBest": float(beamScores[0]),
            "beamMean": float(beamScores.mean()),
        })
        
        # không được thì dừng lại việc thực thi
        if noGainRounds >= maxNoGain:
            break
        
    return bestState, bestScore, history
    


## 4.2 stochasticLocalBeamSearch

Lấy top-k tuyệt đối thì cả `beamWidth` suất thường rơi hết vào con cháu của
cùng một phương án. Beam mất đa dạng, `beamWidth` phương án gần như giống hệt
nhau, thuật toán thoái hoá thành Hill Climbing.

Bốc thăm thì phương án điểm thấp vẫn có cơ hội sống sót. Một cách bố trí trạm
tạm thời kém, sau vài vòng chỉnh sửa, có thể vượt lên dẫn đầu.

In [ ]:
def stochasticLocalBeamSearch(beamWidth, maxRounds=100, maxNoGain=10, seed=0,
                              numStations=None, weights=None):
    rng = np.random.default_rng(seed)
    
    # bốc đại 1 beamWidth phương án để có chỗ bắt đầu
    beam = [randomState(numStations,rng) for _ in range(beamWidth)]
    beamScores = np.array([objective(state,weights) for state in beam])
    
    bestIndex = int(np.argmax(beamScores))
    bestState = beam[bestIndex]
    bestScore = float(beamScores[bestIndex])
    
    history = [{
        "round": 0,
        "bestScore": bestScore,
        "beamBest": float(beamScores.max()),
        "beamMean": float(beamScores.mean()),
    }]

    noGainRounds = 0
    
    for roundNo in range(1,maxRounds+1):
        # gom vào 1 rổ chung
        neighborPool = list({
            newState
            for state in beam
            for newState in getNeighbors(state)
        })

        if not neighborPool:
            break
        
        neighborScores = np.array([objective(state, weights) for state in neighborPool])

        # đổi điểm thành số vé. điểm có thể âm nên dời tất cả lên dương
        # bằng cách trừ đi điểm thấp nhất. cộng baseTickets để phương án bét vẫn còn vé
        scoreSpread = neighborScores.max() - neighborScores.min()
        baseTickets = scoreSpread * 0.1 if scoreSpread > 0 else 1.0
        tickets = neighborScores - neighborScores.min() + baseTickets
        pickProbs = tickets / tickets.sum()
        
        # lấy beamWidth phương thức khác nhau
        pickedIndexes = rng.choice(len(neighborPool), size=beamWidth, replace=False, p=pickProbs)
        beam = [neighborPool[i] for i in pickedIndexes]
        beamScores = neighborScores[pickedIndexes]
        
        # dùng argMax để tìm phương án tốt nhất của vòng
        roundBestIndex = int(np.argmax(beamScores))
        roundBestScore = float(beamScores[roundBestIndex])
        
        if roundBestScore > bestScore:
            bestState = beam[roundBestIndex]
            bestScore = roundBestScore
            noGainRounds = 0
        else:
            noGainRounds += 1

        history.append({
            "round": roundNo,
            "bestScore": bestScore,
            "beamBest": roundBestScore,
            "beamMean": float(beamScores.mean()),
        })
        
        # thực hiện dùng nếu k có kết quả
        if noGainRounds >= maxNoGain:
            break
        
    return bestState, bestScore, history
        
        